# One upload → pairwise + dose-response

Upload a proteomics dataset to Mass Dynamics once, then run **both** analyses off it:

1. **Pairwise**: every compound@dose compared against the vehicle, in a *single* job.
2. **Dose-response**: a 4-parameter curve per protein, in *one job per compound*.

Edit the **Configuration** cell, then run the notebook top to bottom.

### Before you start

```bash
pip install "git+https://github.com/MassDynamics/md-python.git" python-dotenv
```

and put your credentials in a `.env` file next to this notebook:

```
MD_API_BASE_URL=https://app.massdynamics.com/api
MD_AUTH_TOKEN=<your token, no "Bearer " prefix>
```

## Setup

In [ ]:
import csv
import os
from pathlib import Path

from dotenv import load_dotenv
from md_python import MDClient, Dataset, Upload, ExperimentDesign, SampleMetadata

load_dotenv()
assert os.getenv("MD_AUTH_TOKEN"), "MD_AUTH_TOKEN is not set (check your .env)"

client = MDClient(version="v2")
print("API:", client.base_url)
print("Health:", client.health.check())

## Configuration

The only cell you need to edit.

In [ ]:
# Folder with your data files, and which files to upload from it.
DATA_DIR = Path("/Your/Path/To/Data")
FILENAMES = ["YOUR_PROTEIN_FILE.tsv", "YOUR_PEPTIDE_FILE.tsv"]

# Sample metadata CSV: sample_name, condition, dose
METADATA_CSV = Path("/Your/Path/To/Data/sample_metadata.csv")

# Upload name. Must be unique within your organisation.
UPLOAD_NAME = "YOUR UPLOAD NAME"

# spectronaut | diann_tabular | tims_diann | maxquant | md_format | md_format_gene
SOURCE = "md_format"

# Which condition is the vehicle / control. Must match the CSV exactly.
CONTROL_CONDITION = "YOUR_CONTROL_CONDITION"

# Set this to an existing upload id to skip the upload and re-run the analyses.
EXISTING_UPLOAD_ID = None

## 1. Read the sample metadata

Your CSV needs one row per sample and three columns:

| column | meaning |
|---|---|
| `sample_name` | must match the run names inside your data file exactly (for Spectronaut, `R.FileName`) |
| `condition` | the **compound**, not compound+dose. Every dose of a compound shares it. |
| `dose` | a number. The vehicle is `0`. |

```
sample_name,condition,dose
SAMPLE_01,DRUG_1,5
SAMPLE_02,DRUG_1,2
SAMPLE_03,CONTROL,0
```

In [ ]:
with METADATA_CSV.open(newline="", encoding="utf-8-sig") as fh:
    rows = list(csv.DictReader(fh))

for col in ("sample_name", "condition", "dose"):
    assert col in rows[0], f"{METADATA_CSV} is missing a '{col}' column"

conditions = sorted({r["condition"] for r in rows})
print(f"{len(rows)} samples, {len(conditions)} conditions: {conditions}")
rows[:3]

## 2. Build the two metadata tables

**`experiment_design`** maps data-file run names to samples: `[filename, sample_name, condition]`.
For LFQ (one run per sample) `filename` is just the sample name.

**`sample_metadata`** carries the per-sample variables the analyses use. We add a derived
**`condition_dose`** column (e.g. `DRUG_1@5`).

That derived column matters: the pairwise step groups on it so **each dose gets its own comparison
against the vehicle**. Grouping on `condition` alone would pool every dose of a compound into one
group, which is almost never what you want from a dose series.

In [ ]:
experiment_design = ExperimentDesign(
    data=[["filename", "sample_name", "condition"]]
    + [[r["sample_name"], r["sample_name"], r["condition"]] for r in rows]
)

sample_metadata = SampleMetadata(
    data=[["sample_name", "condition", "dose", "condition_dose"]]
    + [
        [r["sample_name"], r["condition"], r["dose"], f"{r['condition']}@{r['dose']}"]
        for r in rows
    ]
)

print(experiment_design)
print()
print(sample_metadata)

## 3. Upload

`client.uploads.create()` blocks while the files transfer, then starts ingestion.

Skipped automatically if you set `EXISTING_UPLOAD_ID` in the config cell.

In [ ]:
if EXISTING_UPLOAD_ID:
    upload_id = EXISTING_UPLOAD_ID
    print("Reusing upload:", upload_id)
else:
    for name in FILENAMES:
        assert (DATA_DIR / name).is_file(), f"Missing data file: {DATA_DIR / name}"

    upload = Upload(
        name=UPLOAD_NAME,
        source=SOURCE,
        experiment_design=experiment_design,
        sample_metadata=sample_metadata,
        file_location=str(DATA_DIR),
        filenames=FILENAMES,
    )
    print(f"Uploading {len(FILENAMES)} file(s)...")
    upload_id = client.uploads.create(upload)
    print("Upload ID:", upload_id)

## 4. Wait for the upload **and** for its dataset

Two waits, not one. The second is the one people miss.

The upload reporting `COMPLETED` does not mean its intensity dataset is `COMPLETED`; they flip a
moment apart. Submit a job into that gap and the pairwise job is rejected outright, while the
dose-response job is *accepted* and then dies server-side with a confusing `NoneType` error. So wait
for the dataset itself before going any further.

In [ ]:
client.uploads.wait_until_complete(upload_id, poll_s=15, timeout_s=7200)
print("Upload ingested.")

dataset = client.datasets.find_initial_dataset(upload_id)
dataset_id = str(dataset.id)

state = client.datasets.wait_until_complete(
    upload_id=upload_id, dataset_id=dataset_id, poll_s=15, timeout_s=7200
)
print("Intensity dataset:", dataset_id, "->", state.state)
assert str(state.state) == "COMPLETED"

## 5. What do the analysis jobs accept?

`GET /jobs` returns each job's parameter definition: field names, types, defaults, and which are
required. This is the source of truth for `job_run_params`, and it is worth a look before you submit:
these definitions change, and what you see here always matches the server you are talking to.

Two notes on reading it:

* `input_datasets` is listed but does **not** go in `job_run_params`. The server fills it in from
  the dataset id you pass separately.
* Some fields only apply in combination, e.g. `filter_threshold_percentage` applies when
  `filter_values_criteria` is `"percentage"`. Send the one that matches.

The `job_get` helper below exists because `jobs.list()` returns plain dicts on some md-python
versions and `Job` objects on others (and spells the published flag `isPublished` in one,
`is_published` in the other). It reads either.

In [ ]:
# Every job slug this API exposes, grouped by run type.
from collections import defaultdict

jobs = client.jobs.list()

by_type = defaultdict(list)
for job in jobs:
    by_type[str(job.run_type)].append(job)

published = sum(1 for j in jobs if j.is_published)
print(f"{len(jobs)} jobs on {client.base_url}  ({published} published)\n")

for run_type in sorted(by_type):
    print(run_type)
    for job in sorted(by_type[run_type], key=lambda j: j.slug or ""):
        flag = "" if job.is_published else "   <- not published"
        print(f"    {str(job.slug):<32} {len(job.properties or {}):>3} params   {job.name}{flag}")
    print()

# Bare list. Paste into a comment, or use it to validate a job_slug before submitting
print(sorted(j.slug for j in jobs if j.is_published and j.slug))


In [ ]:
def show_params(slug):
    """Print the parameters a job accepts. Fetches its own job list, so this cell
    works on its own, with no dependency on an earlier cell having run."""
    job = next((j for j in client.jobs.list() if j.slug == slug), None)
    assert job, f"No job with slug {slug!r}"

    props = job.properties or {}
    print(f"{slug}: {len(props)} parameters\n")
    for name, spec in props.items():
        if not isinstance(spec, dict):
            print(f"  {name}")
            continue
        bits = []
        if spec.get("fieldType"):
            bits.append(str(spec["fieldType"]))
        if "default" in spec:
            bits.append(f"default={spec['default']!r}")
        if any(r.get("name") == "is_required" for r in spec.get("rules", [])):
            bits.append("REQUIRED")
        print(f"  {name:<32} {'  '.join(bits)}")

show_params("pairwise_comparison")


## 6. Pairwise: every compound@dose vs the vehicle

All the comparisons go into **one** job on purpose: limma then computes a single shared FDR
correction across them. Do not loop and submit one job per pair.

In [ ]:
design_columns = sample_metadata.to_columns()
CONDITION_COLUMN = "condition_dose"   # or "condition" to pool doses

groups = []
for value in design_columns[CONDITION_COLUMN]:
    if value not in groups:
        groups.append(value)

control_groups = {
    g for g, cond in zip(design_columns[CONDITION_COLUMN], design_columns["condition"])
    if cond == CONTROL_CONDITION
}
assert control_groups, f"No samples with condition == {CONTROL_CONDITION!r}"
assert len(control_groups) == 1, (
    f"The vehicle spans several groups: {sorted(control_groups)}. "
    "Give it a single dose value, or set CONDITION_COLUMN = 'condition'."
)
control = control_groups.pop()

comparisons = [[g, control] for g in groups if g != control]
print(f"{len(comparisons)} comparisons against {control!r}")
comparisons[:5]

The two parameters below are the ones that are easy to get wrong. Both were taken from the job
spec printed above, not guessed:

* **`filter_values_criteria`** is a **string** (`"percentage"` or `"count"`), *not* a dict, and it
  needs a matching sibling: `filter_threshold_percentage` or `filter_threshold_count`.
* **`de_method_<entity_type>`** is **required** and is keyed by entity type, so protein data needs
  `de_method_protein`.

In [ ]:
ENTITY_TYPE = "protein"   # protein | peptide | gene

pairwise_params = {
    "condition_column": CONDITION_COLUMN,
    "condition_comparisons": {"condition_comparison_pairs": comparisons},
    "experiment_design": design_columns,
    "entity_type": ENTITY_TYPE,
    f"de_method_{ENTITY_TYPE}": "limma",
    "filter_values_criteria": "percentage",
    "filter_threshold_percentage": 0.5,
    "filter_valid_values_logic": "at least one condition",
    "fit_separate_models": True,
    "limma_trend": True,
    "robust_empirical_bayes": True,
}

pairwise_id = client.datasets.create(
    Dataset(
        input_dataset_ids=[dataset_id],
        name=f"{UPLOAD_NAME} - pairwise vs control",
        job_slug="pairwise_comparison",
        job_run_params=pairwise_params,
    )
)
print("Pairwise dataset:", pairwise_id)

## 7. Dose-response: one job per compound

The opposite fan-out to pairwise, and for a concrete reason: the dose-response pipeline fits one
curve per protein and groups samples by **dose only**. It has no notion of which compound a sample
came from, so putting two compounds in one job silently pools them into a single meaningless curve.

Each job therefore gets one compound's dose series **plus the vehicle samples** as the dose-0 anchor.

The platform requires at least **5 samples** and **3 distinct dose levels** per job (the vehicle's 0
counts). Compounds that fall short are reported and skipped rather than failing the run.

In [ ]:
control_rows = [r for r in rows if r["condition"] == CONTROL_CONDITION]
compounds = [c for c in conditions if c != CONTROL_CONDITION]

dose_response_ids = {}
for compound in compounds:
    subset = [r for r in rows if r["condition"] in (compound, CONTROL_CONDITION)]
    sample_names = [r["sample_name"] for r in subset]
    doses = [float(r["dose"]) for r in subset]      # must be numbers, not strings
    distinct = sorted(set(doses))

    if len(sample_names) < 5 or len(distinct) < 3:
        print(f"  SKIP {compound}: {len(sample_names)} samples, {len(distinct)} dose levels")
        continue

    # experiment_design is the ONLY thing that scopes the job.
    params = {
        "experiment_design": {"sample_name": sample_names, "dose": doses},
        "control_samples": [r["sample_name"] for r in control_rows],
        "log_intensities": True,
        "use_imputed_intensities": False,
        "normalise": "none",
        "span_rollmean_k": 1,
        "prop_required_in_protein": 0.5,
    }

    ds_id = client.datasets.create(
        Dataset(
            input_dataset_ids=[dataset_id],
            name=f"{UPLOAD_NAME} - DR {compound}",
            job_slug="dose_response",
            job_run_params=params,
        )
    )
    dose_response_ids[compound] = ds_id
    print(f"  {compound}: {len(sample_names)} samples, doses {distinct} -> {ds_id}")

## 8. Wait for everything

In [ ]:
pw_state = client.datasets.wait_until_complete(
    upload_id=upload_id, dataset_id=pairwise_id, poll_s=15, timeout_s=7200
)
print("pairwise:", pw_state.state)

for compound, ds_id in dose_response_ids.items():
    st = client.datasets.wait_until_complete(
        upload_id=upload_id, dataset_id=ds_id, poll_s=15, timeout_s=7200
    )
    print(f"{compound}: {st.state}")

## Results

Everything is now visible in the Mass Dynamics web app under this upload. To pull a table down
programmatically:

```python
client.datasets.list_by_upload(upload_id)                       # what exists
[t.name for t in client.datasets.get_by_id(pairwise_id).tables] # what tables it has
url = client.datasets.download_table_url(pairwise_id, "output_comparisons", format="csv")
```

| dataset type | tables |
|---|---|
| `PAIRWISE` | `output_comparisons`, `runtime_metadata` |
| `DOSE_RESPONSE` | `output_curves`, `output_volcanoes`, `input_drc`, `runtime_metadata` |

In `output_volcanoes` the half-maximal dose column is **`ED50`** (the `drc` package's terminology),
not `EC50`.

### If a job is rejected

Re-run the `show_params(...)` cell and compare the field list against what you are sending. The
API is the source of truth and these definitions do change.